# Annotation listing and navigation

This notebook imports pUC19 from a GenBank file, loads its feature annotations,
and navigates to the MCS (multiple cloning site) using `widget.go_to()` and
`widget.show()`.

In [1]:
import os
import tempfile
import gen

## Import pUC19

Create a fresh in-memory repository and import the GenBank fixture.

In [2]:
# Adjust this path if running from outside the project root.
FIXTURE = os.path.abspath("../../fixtures/puc19.gb")

tmp = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmp, ".gen"))
repo.import_genbank(FIXTURE)

bg = repo.get_block_groups()[0]
print("Block group:", bg.name)

Block group: sequence-222046-


## Plot the graph

Render the sequence graph in a widget. Annotation groups associated with the block group are automatically loaded as track panels on creation.

In [3]:
widget = bg.plot(rows=24)

# Annotation groups for this block group are auto-loaded on creation.
anns = widget.list_annotations()
print(f"{len(anns)} annotations loaded")
widget

21 annotations loaded


## Annotation files

Additional GFF3 or BED annotation files can be loaded at any time:

```python
widget.add_track_file("path/to/features.gff3")
```

To start with a blank canvas before loading your own files, clear the auto-loaded tracks:

```python
widget.clear_all_annotations()
```

## List all annotations

`widget.list_annotations()` returns `Annotation` objects spanning all
sources (track panels and inline highlights).  Each annotation exposes `.name`,
`.locus`, and can be passed directly to `widget.go_to()` or `widget.show()`.

In [4]:
anns = widget.list_annotations()
print(f"{len(anns)} annotations loaded")
for a in anns:
    print(a)

21 annotations loaded
Annotation(name="source", segments=1)
Annotation(name="pBR322ori-F", segments=1)
Annotation(name="L4440", segments=1)
Annotation(name="CAP binding site", segments=1)
Annotation(name="lac promoter", segments=3)
Annotation(name="lac operator", segments=1)
Annotation(name="M13/pUC Reverse", segments=1)
Annotation(name="M13 rev", segments=1)
Annotation(name="M13 Reverse", segments=1)
Annotation(name="lacZ-alpha", segments=1)
Annotation(name="MCS", segments=1)
Annotation(name="M13 Forward", segments=1)
Annotation(name="M13 fwd", segments=1)
Annotation(name="M13/pUC Forward", segments=1)
Annotation(name="pRS-marker", segments=1)
Annotation(name="pGEX 3'", segments=1)
Annotation(name="pBRforEco", segments=1)
Annotation(name="AmpR promoter", segments=1)
Annotation(name="AmpR", segments=2)
Annotation(name="Amp-R", segments=1)
Annotation(name="ori", segments=2)


## Navigate to the MCS

Filter for the MCS feature and navigate to it two ways:

* `widget.go_to(ann)` — left-pins the annotation start at column 12, no highlight
* `widget.show(ann)` — left-pins the annotation start and adds a highlight

In [5]:
mcs = next(a for a in anns if a.name == "MCS")
print("MCS locus:", mcs.locus)
print("MCS start:", mcs.locus.start())

MCS locus: [348d9388:0-2686:631-688]
MCS start: GraphPos(348d9388[0..2686] +631)


In [6]:
# Left-pin the MCS start at column 12 from the left edge.
widget.go_to(mcs)
widget

In [7]:
# Centre on the MCS and add a highlight.
widget.show(mcs)
widget

## Filter and batch-navigate

You can filter the list and pass any annotation directly to `go_to` or `show`.

In [8]:
promoters = [a for a in anns if "promoter" in a.name.lower()]
print("Promoters:", [a.name for a in promoters])

if promoters:
    widget.show(promoters[0])
    widget

Promoters: ['lac promoter', 'AmpR promoter']


## Search and navigate

Search for a sequence, wrap matches as `Annotation` objects, then navigate to one.

In [9]:
# Search for the lac operator sequence and add matches to the widget.
results = repo.search("AATTGTGAGCGGATAACAATT")
matches = [locus for _, loci in results for locus in loci]
print(f"{len(matches)} match(es) found")

for i, locus in enumerate(matches):
    widget.add_annotation(gen.Annotation(locus, f"lac-op-hit ({i + 1} of {len(matches)})"))

1 match(es) found


In [10]:
# List all annotations now in the widget, then navigate to the first hit.
anns2 = widget.list_annotations()
print(f"{len(anns2)} total annotations")

hits = [a for a in anns2 if a.name.startswith("lac-op-hit")]
print(f"{len(hits)} lac operator hit(s)")

widget.go_to(hits[0])
widget

24 total annotations
1 lac operator hit(s)
